# Capítulo 3 - Exercício 4: Criando um detector de Spam

O objetivo deste notebook é treinar um detector de Spam.

## Plano do exercício

1. Carregar os dados do Spam e do Ham.
2. Separar os dados em treino e teste usando a divisão padrão do dataset.
3. Treinar um classificador binário para detectar o dígito 5.
4. Medir uma acurácia inicial com validação cruzada.
5. Usar `GridSearchCV` para escolher melhores hiperparâmetros.
6. Repetir o processo para o problema multiclasse, classificando todos os dígitos.
7. Avaliar o melhor modelo final no conjunto de teste.

## Configuração

Importamos as bibliotecas usadas no exercício e verificamos versões mínimas. Mantemos uma semente fixa para tornar os resultados mais reprodutíveis.

In [1]:
import sys
import numpy as np

from packaging import version
import sklearn

assert sys.version_info >= (3, 7)
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

np.random.seed(42)

# Spam e Ham

## Importando os dados

In [2]:
from pathlib import Path

def import_dados(path):
    dados = []
    for email in sorted(Path(path).iterdir()):
        if email.is_file() and email.name[0].isdigit():
            conteudo = email.read_text(encoding="utf-8", errors="replace")
            dados.append(conteudo)
    return dados

easy_ham = import_dados('./emails/easy_ham')
spam_1 = import_dados('./emails/spam_1')


## Dividindo os dados

In [3]:
train_ham, test_ham = easy_ham[:2000], easy_ham[2000:] 
train_spam, test_spam = spam_1[:400], spam_1[400:] 
print(len(train_ham))
print(len(test_ham))
print(len(train_spam))
print(len(test_spam))

2000
551
400
100


## Pipeline modular de pre-processamento

A ideia agora e separar cada tema em uma celula pequena. Isso deixa o codigo mais explicavel e tambem prepara o caminho para usar exatamente o mesmo pre-processamento no treino e no teste.

O fluxo que vamos construir e:

1. montar `X_raw` e `y` com os e-mails brutos;
2. dividir treino/teste com estratificacao;
3. aplicar um extrator de features nos e-mails de treino;
4. aplicar o mesmo extrator nos e-mails de teste;
5. depois conectar esse extrator a um modelo do scikit-learn.


### 1. Imports e padroes compartilhados

Esta celula concentra Bibliotecas e expressoes regulares usadas por varias partes da pipeline

Se voce quiser melhorar a detecao de palavras promocionais, este eh um dos lugares certos para mexer


In [4]:
import re 
from email import policy
from email.parser import Parser
from email.utils import getaddresses
from html.parser import HTMLParser
from urllib.parse import urlparse

import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split

URL_RE = re.compile(r"https?://[^\s<>\"')]+|www\.[^\s<>\"')]+", re.IGNORECASE)

MONEY_RE = re.compile(
    r"(?:R\$|US\$|\$|€|£)\s?\d+(?:[.,]\d+)*|\b\d+(?:[.,]\d+)?\s?(?:%|percent|dollars?|usd|reais)\b",
    re.IGNORECASE,
)

PROMO_RE = re.compile(
     r"\b(?:free|bonus|offer|discount|guarantee|guaranteed|winner|win|cash|money|save|order now|click here|limited|urgent|risk free|no risk)\b",
    re.IGNORECASE,
)

### 2. HTML para TEXTO

Spam costuma usar HTML para layout, cor fontes grandes e botoes

Para o modelo, o mais util normalmente e transformar esse HTML em texto limpo, preservando a mensagem e removendo as tags

In [ ]:
class HTMLTextExtractor(HTMLParser):

    def __init__(self):
        super().__init__()
        self.parts = []
        self.skip_depth = 0

    def handle_starttag(self, tag, _attrs):
        tag = tag.lower()
        if tag in {"script","style"}:
            self.skip_depth += 1
        elif tag in {"br","p","div","tr","li"}:
            self.parts.append(" ")
    
    def handle_endtag(self, tag):
        if tag.lower() in {"script", "style"} and self.skip_depth:
            self.skip_depth -= 1

    def handle_data(self, data):
        if not self.skip_depth:
            self.parts.append(data)

    def text(self):
        return re.sub(r"\s+", " ", " ".join(self.parts)).strip()
    
def html_para_texto(html):
    parser = HTMLTextExtractor()
    parser.feed(html or "")
    return parser.text()
